# TRL Results — Load Checkpoint & Plot

Loads a saved TRL checkpoint for `pointmaze-large-navigate-v0` and regenerates the diagnostic plots from Section 6 of the training notebook (training curves, per-task success, Q-value distribution).

**Assumes:**
- Checkpoint saved via `save_agent(agent, SAVE_DIR, step)` (a `params_{step}.pkl` lives in `SAVE_DIR`).
- Training/eval history pickled to `SAVE_DIR/history.pkl` at the end of the training run — see the **Load history** cell below for the expected format and a `wandb` alternative.


## 1. Setup

In [ ]:
!pip install -q \
    "jax[cuda12]>=0.4.26" \
    "flax>=0.8.4" \
    "distrax>=0.1.5" \
    ml_collections \
    matplotlib \
    moviepy \
    wandb \
    ogbench \
    imageio \
    tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 704.8/704.8 MB 861.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 

In [ ]:
import os
import pickle
import random
from collections import defaultdict

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

os.environ.setdefault('MUJOCO_GL', 'egl')

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cd /content/drive/MyDrive/cos435-final-project/test-runs/Transitive_Reinforcement_Learning_COS435/ogbench-master/impls/

/content/drive/MyDrive/cos435-final-project/test-runs/Transitive_Reinforcement_Learning_COS435/ogbench-master/impls


In [ ]:
# Paths / checkpoint selection — edit these for your run.
ENV_NAME      = 'pointmaze-large-navigate-v0'
SEED          = 0
DRIVE_ROOT    = '/content/drive/MyDrive/cos435-final-project/test-runs'
SAVE_DIR      = f'{DRIVE_ROOT}/checkpoints_alpha10/{ENV_NAME}_seed{SEED}'
RESTORE_STEP  = 1_000_000
HISTORY_PATH  = f'{SAVE_DIR}/history.pkl'


## 2. Load env + config, rebuild agent skeleton, restore checkpoint

We need to rebuild the agent with the same `config` used at training time so the parameter PyTree matches the saved checkpoint. The env + dataset are also loaded because (a) `TRLDataset` gives us a batch to initialize the agent's network shapes, and (b) we need `task_infos` and a batch for the Q-value plot at the end.


In [ ]:
from agents import agents
from agents.trl import get_config
from utils.datasets import Dataset, TRLDataset
from utils.env_utils import make_env_and_datasets

config = get_config()
config['discount']               = 0.99
config['alpha']                  = 10.0
config['distance_weight_lambda'] = 0.7
assert config['policy_extraction'] == 'ddpgbc'

env, train_dataset, val_dataset = make_env_and_datasets(ENV_NAME, frame_stack=config['frame_stack'])
train_dataset_trl = TRLDataset(Dataset.create(**train_dataset), config)

task_infos = env.unwrapped.task_infos if hasattr(env.unwrapped, 'task_infos') else env.task_infos
num_tasks  = len(task_infos)
print(f'{ENV_NAME}: {num_tasks} eval tasks, {len(train_dataset_trl.lengths)} train trajectories')


pointmaze-large-navigate-v0.npz: 100%|██████████| 19.5M/19.5M [00:00<00:00, 20.9MB/s]


pointmaze-large-navigate-v0-val.npz: 100%|██████████| 1.95M/1.95M [00:00<00:00, 4.84MB/s]


pointmaze-large-navigate-v0: 5 eval tasks, 2000 train trajectories


In [ ]:
from utils.flax_utils import restore_agent

random.seed(SEED); np.random.seed(SEED)

example_batch = train_dataset_trl.sample(1)
agent_class   = agents[config['agent_name']]
agent = agent_class.create(
    SEED,
    example_batch['observations'],
    example_batch['actions'],
    config,
)
agent = restore_agent(agent, SAVE_DIR, restore_epoch=RESTORE_STEP)
print(f'Restored agent from {SAVE_DIR} at step {RESTORE_STEP:,}')


ERROR:2026-04-23 23:20:10,936:jax._src.xla_bridge:487: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/xla_bridge.py", line 485, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/usr/local/lib/python3.12/dist-packages/jax_plugins/xla_cuda12/__init__.py", line 328, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/usr/local/lib/python3.12/dist-packages/jax_plugins/xla_cuda12/__init__.py", line 285, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: Unknown CUDA error 303; cuGetErrorName failed. This probably means that JAX was unable to load the CUDA libraries.
ERROR:jax._src.xla_bridge:Jax plugin configuration error: Exception when calling jax_plu

Restored from /content/drive/MyDrive/cos435-final-project/test-runs/checkpoints_alpha10/pointmaze-large-navigate-v0_seed0/params_1000000.pkl
Restored agent from /content/drive/MyDrive/cos435-final-project/test-runs/checkpoints_alpha10/pointmaze-large-navigate-v0_seed0 at step 1,000,000


## 3. Load training / eval history

The training loop populates `train_history` and `eval_history` as `defaultdict(list)` keyed by metric name, with each value a list of `(step, value)` tuples. To plot after the fact, dump them at the end of the training run:

```python
import pickle
with open(f'{SAVE_DIR}/history.pkl', 'wb') as f:
    pickle.dump({'train': dict(train_history), 'eval': dict(eval_history)}, f)
```

If you only logged to wandb, use the alternative cell below instead.


In [ ]:
# Option A: load from pickled history.
with open(HISTORY_PATH, 'rb') as f:
    histories = pickle.load(f)
train_history = defaultdict(list, histories['train'])
eval_history  = defaultdict(list, histories['eval'])
print(f'Loaded {len(train_history)} train metrics, {len(eval_history)} eval metrics')


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/cos435-final-project/test-runs/checkpoints_alpha10/pointmaze-large-navigate-v0_seed0/history.pkl'

In [ ]:
# Option B (alternative): load from wandb run history.
# Uncomment and fill in WANDB_PATH = '<entity>/<project>/<run_id>'.
#
# import wandb
# api = wandb.Api()
# run = api.run(WANDB_PATH)
# df  = run.history(samples=10_000, pandas=True)
#
# train_history = defaultdict(list)
# eval_history  = defaultdict(list)
# step_col = '_step' if '_step' in df.columns else 'step'
# for col in df.columns:
#     if col.startswith('_') or col == step_col:
#         continue
#     sub = df[[step_col, col]].dropna()
#     pts = list(zip(sub[step_col].astype(int).tolist(), sub[col].astype(float).tolist()))
#     if col.startswith('success/'):
#         eval_history[col] = pts
#     else:
#         train_history[col] = pts
# print(f'From wandb: {len(train_history)} train metrics, {len(eval_history)} eval metrics')


## 4. Training curves (critic/actor losses, pred vs target, overall success)

In [ ]:
def series(history, key):
    pts = history[key]
    if not pts:
        return np.array([]), np.array([])
    xs, ys = zip(*pts)
    return np.array(xs), np.array(ys)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Critic loss
x, y = series(train_history, 'critic/critic_loss')
axes[0, 0].plot(x, y)
axes[0, 0].set_title('Critic loss')
axes[0, 0].set_xlabel('step'); axes[0, 0].set_ylabel('loss')

# Actor loss
x, y = series(train_history, 'actor/actor_loss')
axes[0, 1].plot(x, y)
axes[0, 1].set_title('Actor loss')
axes[0, 1].set_xlabel('step')

# Pred vs target mean
x_p, y_p = series(train_history, 'critic/pred_mean')
x_t, y_t = series(train_history, 'critic/target_mean')
axes[1, 0].plot(x_p, y_p, label='pred_mean')
axes[1, 0].plot(x_t, y_t, label='target_mean')
axes[1, 0].set_title('Critic predictions vs targets')
axes[1, 0].set_xlabel('step'); axes[1, 0].set_ylim(0, 1); axes[1, 0].legend()

# Overall success
x, y = series(eval_history, 'success/overall')
axes[1, 1].plot(x, y, marker='o')
axes[1, 1].set_title('Overall success rate')
axes[1, 1].set_xlabel('step'); axes[1, 1].set_ylim(-0.05, 1.05)

plt.tight_layout(); plt.show()


## 5. Per-task success

Made more square than the original `(10, 4)` layout so it sits nicely next to other panels in the writeup.


In [ ]:
per_task_keys = sorted(k for k in eval_history if k.startswith('success/') and k != 'success/overall')

plt.figure(figsize=(6, 6))  # square-ish; was (10, 4)
for k in per_task_keys:
    x, y = series(eval_history, k)
    plt.plot(x, y, marker='.', label=k.replace('success/', ''))
plt.title('Per-task success')
plt.xlabel('step'); plt.ylabel('success')
plt.ylim(-0.05, 1.05)
plt.legend(fontsize=8, loc='best')
plt.tight_layout(); plt.show()


## 6. Q-value distribution from the loaded checkpoint

Sanity check on the learned value function: on a fresh batch of `(s_i, a_i, s_j)` triples we expect Q values spread across `(0, 1)` (sigmoid-squashed) and the implied distances `log_γ Q` to span a plausible horizon.


In [ ]:
batch  = train_dataset_trl.sample(4096)
logits = agent.network.select('critic')(batch['s_i'], batch['s_j'], batch['a_i'])
if logits.ndim > 1:
    logits = jnp.min(logits, axis=0)
probs     = jax.nn.sigmoid(logits)
distances = jnp.log(jnp.clip(probs, 1e-8, 1 - 1e-8)) / jnp.log(config['discount'])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(np.array(probs), bins=50)
axes[0].set_title('Q(s_i, a_i, s_j) distribution'); axes[0].set_xlabel('Q'); axes[0].set_xlim(0, 1)
axes[1].hist(np.array(distances), bins=50)
axes[1].set_title(r'Estimated distance $\log_\gamma Q$'); axes[1].set_xlabel('steps')
plt.tight_layout(); plt.show()

print(f'Q summary: min={float(probs.min()):.3f} '
      f'median={float(jnp.median(probs)):.3f} '
      f'max={float(probs.max()):.3f}')
